# 04 변이·복제수·RPPA 표
03의 `gdc_files.csv`에서 코호트 환자의 원발 종양 파일(MAF, Gene Level Copy Number, RPPA)만 GDC에서 받아 분석용 표로 만든다. 병리 이미지는 받지 않는다.
- GDC harmonized(hg38) 자료를 쓴다. 기존 RNA·메틸화와 같은 출처이고, 파일 ID를 03에서 이미 확보했다.
- 커널: `C:\Users\Admin\EXAONEPath\exaonepath-venv` (인터넷 필요)
- 출력: `C:\Users\Admin\EXAONEPath_data\v4\04_pancan_tables\{raw\, maf_nonsyn.parquet, cnv_gene.parquet, rppa.parquet, files_used.csv}`

In [3]:
# 1. 경로와 대상 파일 (환자당 층별 1개)
from pathlib import Path
import requests, pandas as pd

OUT2 = Path(r"C:\Users\Admin\EXAONEPath_data\v4\02_cohort")
OUT3 = Path(r"C:\Users\Admin\EXAONEPath_data\v4\03_gdc_inventory")
OUT  = Path(r"C:\Users\Admin\EXAONEPath_data\v4\04_pancan_tables")
RAW  = OUT / "raw"
OUT.mkdir(parents=True, exist_ok=True)
co = pd.read_csv(OUT2 / "cohort.csv").query("excluded.isna() or excluded == ''")

f = pd.read_csv(OUT3 / "gdc_files.csv")
f = f[f.patient_id.isin(co.patient_id) & (f.sample_type == "Primary Tumor") & f.layer.isin(["MAF", "CNV", "RPPA"])].copy()
PREF = ["ascat3", "ascat2", "absolute"]  # CNV 워크플로 우선순위 (파일명 기준)
f["rank"] = f.file_name.str.lower().map(lambda n: next((i for i, k in enumerate(PREF) if k in n), 9))
f = f.sort_values(["layer", "patient_id", "rank", "file_name"]).drop_duplicates(["layer", "patient_id"])
f.to_csv(OUT / "files_used.csv", index=False, encoding="utf-8-sig")
print(f.groupby(["layer", "cancer"]).size().unstack())
print("CNV 워크플로:", f[f.layer == "CNV"]["rank"].map(dict(enumerate(PREF))).value_counts(dropna=False).to_dict())

cancer  LUAD  LUSC
layer             
CNV      402   347
MAF      398   341
RPPA     279   238
CNV 워크플로: {'ascat3': 725, 'absolute': 23, nan: 1}


In [ ]:
# 2. 다운로드: 파일 하나씩, 4개 병렬, 실패 시 재시도 (이미 받은 파일은 건너뜀, 다시 실행하면 이어서 받음)
import time
from concurrent.futures import ThreadPoolExecutor

def get(r, tries=5):
    p = RAW / r.layer / r.file_id / r.file_name
    if p.exists(): return None
    for t in range(tries):
        try:
            x = requests.get(f"https://api.gdc.cancer.gov/data/{r.file_id}", timeout=300)
            x.raise_for_status()
            p.parent.mkdir(parents=True, exist_ok=True)
            tmp = p.with_name(p.name + ".part"); tmp.write_bytes(x.content); tmp.replace(p)
            return None
        except requests.RequestException:
            time.sleep(15 * (t + 1))
    return r.file_id

with ThreadPoolExecutor(4) as ex:
    fail = [i for i in ex.map(get, f.itertuples()) if i]
done = sum((RAW / r.layer / r.file_id / r.file_name).exists() for r in f.itertuples())
print(f"완료 {done}/{len(f)}, 실패 {len(fail)}")

In [ ]:
# 3. 표 만들기
path = lambda r: RAW / r.layer / r.file_id / r.file_name
NONSYN = {"Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del", "Frame_Shift_Ins", "In_Frame_Del",
          "In_Frame_Ins", "Splice_Site", "Translation_Start_Site", "Nonstop_Mutation"}

maf = pd.concat([pd.read_csv(path(r), sep="\t", comment="#", low_memory=False,
                             usecols=["Hugo_Symbol", "Variant_Classification", "HGVSp_Short"]).assign(patient_id=r.patient_id)
                 for r in f[f.layer == "MAF"].itertuples()])
maf = maf[maf.Variant_Classification.isin(NONSYN)]
maf.to_parquet(OUT / "maf_nonsyn.parquet", index=False)

cnv = pd.concat({r.patient_id: pd.read_csv(path(r), sep="\t", usecols=["gene_name", "copy_number"])
                 .groupby("gene_name").copy_number.mean() for r in f[f.layer == "CNV"].itertuples()}, axis=1)
cnv.astype("float32").to_parquet(OUT / "cnv_gene.parquet")

rppa = pd.concat({r.patient_id: pd.read_csv(path(r), sep="\t").groupby("peptide_target").protein_expression.mean()
                  for r in f[f.layer == "RPPA"].itertuples()}, axis=1)
rppa.to_parquet(OUT / "rppa.parquet")

print("MAF 비동의 변이", len(maf), "건 /", maf.patient_id.nunique(), "명")
print("CNV", cnv.shape, "(유전자 × 환자)  RPPA", rppa.shape, "(단백 × 환자)")